# Laboratory Work: Data Preparation for Machine Learning

### Goal

Learn the main steps of preparing a dataset before training a machine learning model.

In this laboratory work we will practice:

- inspecting a dataset;
- identifying duplicate samples and redundant features;
- finding and imputing missing values;
- identifying outliers;
- transforming numerical variables;
- encoding categorical variables;
- splitting data into training and test sets;
- fitting preprocessing **only on the training data**;
- combining preprocessing and classification in a `Pipeline`.

The dataset is based on the **Breast Cancer Wisconsin** dataset from `scikit-learn`, but it has been intentionally modified to contain common data-quality problems.

> Main idea: **good machine learning starts with good data preparation**.


## 0. Import libraries

If a package is missing, uncomment the installation line.


In [ ]:
# If needed, uncomment:
# !pip install numpy pandas matplotlib scikit-learn scipy mrmr-selection

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    RobustScaler,
    PowerTransformer,
    OneHotEncoder,
    OrdinalEncoder,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    ConfusionMatrixDisplay,
)
from sklearn.base import clone
from mrmr import mrmr_classif

RANDOM_STATE = 42


## 1. Load the dataset

Load the CSV from the course repository (internet required), or replace the URL with a local CSV path.


In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/vytkuc/inf4039_2026_autumn/refs/heads/main/lab01_data_prep/messy_breast_cancer_data.csv")

print("Dataset shape:", df.shape)
df.head()


## 2. Initial inspection

Before changing anything, first understand the dataset.

Useful questions:

- How many samples and features are there?
- What are the data types?
- Are there missing values?
- Are there duplicate rows?
- Do some features contain only one unique value?


In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isna().sum().sort_values(ascending=False))

print("\nNumber of duplicate rows:", df.duplicated().sum())

print("\nNumber of unique values per column:")
print(df.nunique(dropna=False).sort_values())


In [ ]:
# Basic descriptive statistics for numerical variables
df.describe().T


### Pandas essentials
A Series is one labelled column; a DataFrame is a table. Work on copies for demonstrations.

- `loc`: label-based selection; label slice endpoints are included.
- `iloc`: position-based selection; the stop position is excluded.
- `at` / `iat`: access one scalar by labels / positions.
- `concat(axis=0)` stacks rows; `concat(axis=1)` aligns columns by index.


In [ ]:
fruit = pd.DataFrame({"apples": [3, 2, 0], "oranges": [0, 3, 7]})
print(type(fruit), type(fruit["apples"]))
display(df.tail())
df.info()
display(df["target"].value_counts())
display(df.loc[0:4, ["mean radius", "target"]])
display(df.iloc[:5, :3])
print(df.at[0, "mean radius"], df.iat[0, 0])

demo = df.rename(columns={"mean radius": "mean_radius"}).copy()
demo = demo.drop(columns=["constant_feature"])
demo["target_label"] = demo["target"].replace({0: "malignant", 1: "benign"})
display(demo.head())
display(pd.concat([df.head(2), df.tail(2)], axis=0))
display(pd.concat([df["mean radius"], df["mean area"]], axis=1).head())
print("Mean:", df["mean radius"].mean())
print("Median:", df["mean radius"].median())
print("Modes:", df["device"].mode().tolist())
numeric_matrix = df[["mean radius", "mean area"]].to_numpy()
print("NumPy matrix:", numeric_matrix.shape, numeric_matrix.dtype)

# A small example of invalid numeric entries.
text_numbers = pd.Series(["21", "35", "not recorded"])
display(pd.to_numeric(text_numbers, errors="coerce"))


### Visual inspection

Boxplots are useful for spotting unusually large or small values.


In [ ]:
for column in ["mean radius", "mean texture", "mean area"]:
    plt.figure(figsize=(7, 3))
    df[[column]].boxplot()
    plt.title(f"Boxplot: {column}")
    plt.ylabel(column)
    plt.tight_layout()
    plt.show()


## 3. Redundant samples and redundant features

Two simple problems are:

1. **Duplicate samples** – the same row appears more than once.
2. **Zero-variance features** – a feature contains the same value for every sample.

We remove exact duplicate rows before the train/test split so that the same sample cannot appear in both datasets.


In [ ]:
df_clean = df.copy()

print("Rows before removing duplicates:", len(df_clean))
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print("Rows after removing duplicates:", len(df_clean))

zero_variance_features = [
    col for col in df_clean.columns
    if col != "target" and df_clean[col].nunique(dropna=True) <= 1
]

print("\nZero-variance features:", zero_variance_features)

df_clean = df_clean.drop(columns=zero_variance_features)

print("Shape after basic cleaning:", df_clean.shape)


## 4. Split the data into training and test sets

The target variable is `target`:

- `0` – malignant;
- `1` – benign.

A key rule is:

> **Split first. Fit data-preparation parameters on the training set only. Then apply the same transformations to both training and test data.**

This helps prevent **data leakage**.


In [ ]:
X = df_clean.drop(columns="target")
y = df_clean["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))
print("\nTraining class counts:")
print(y_train.value_counts().sort_index())


### 4.1 Exploratory plots and correlations
Use the training subset for feature-related exploration in the predictive workflow.

**Pearson** measures linear association. Normality is not required to calculate it; distributional assumptions matter for some inferential tests. There is no universal rule requiring more than 30 samples.
**Spearman** measures monotonic association using ranks; it can be used with either normally or non-normally distributed data.
**Kendall tau-b** measures rank concordance and handles ties. Rank correlations need meaningful ordering; arbitrary nominal category codes are not suitable.

Correlation is not causation. Missing values are excluded pairwise, so pairs can have different sample counts. Strong predictor–predictor correlations suggest redundancy, but do not automatically justify deletion.


In [ ]:
eda_features = ["mean radius", "mean texture", "mean perimeter", "mean area"]
eda = X_train[eda_features].copy()
eda["target"] = y_train

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(eda["mean radius"].dropna(), bins=25)
axes[0].set(xlabel="Mean radius", ylabel="Count", title="Distribution")
for label, name in [(0, "malignant"), (1, "benign")]:
    group = eda.loc[eda["target"] == label]
    axes[1].scatter(group["mean radius"], group["mean area"], label=name, alpha=0.6)
axes[1].set(xlabel="Mean radius", ylabel="Mean area", title="Relationship")
axes[1].legend()
eda.boxplot(column="mean radius", by="target", ax=axes[2])
axes[2].set(title="Radius by class", xlabel="0 malignant / 1 benign")
fig.suptitle("")
fig.tight_layout()
OUTPUT_DIR = Path("lab01_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
fig.savefig(OUTPUT_DIR / "exploratory_plots.png", dpi=150, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
correlation_matrices = {}
for ax, method in zip(axes, ["pearson", "spearman", "kendall"]):
    corr = eda.corr(method=method)
    correlation_matrices[method] = corr
    image = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr)), corr.columns, rotation=90)
    ax.set_yticks(range(len(corr)), corr.index)
    ax.set_title(method.title())
    fig.colorbar(image, ax=ax, shrink=0.7)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "correlations.png", dpi=150, bbox_inches="tight")
plt.show()
display(pd.DataFrame({m: c["target"] for m, c in correlation_matrices.items()}))
print("One coefficient:", correlation_matrices["pearson"].at["mean radius", "mean area"])


## 5. Missing values

Common options for missing values include:

- removal;
- mean imputation;
- median imputation;
- most-frequent-value imputation;
- constant-value imputation;
- KNN imputation.

For numerical variables, the **median** is often useful when outliers are present.
For categorical variables, the **most frequent value** is a simple choice.


In [ ]:
print("Missing values in the training set:")
X_train.isna().sum().sort_values(ascending=False)


### 5.1 Simple imputation example

Here we impute one numerical feature using the training-set median.


In [ ]:
example_feature = "mean radius"

median_imputer = SimpleImputer(strategy="median")

train_radius = X_train[[example_feature]]
train_radius_imputed = median_imputer.fit_transform(train_radius)

print("Training-set median used for imputation:",
      float(median_imputer.statistics_[0]))

print("Missing values before:", train_radius.isna().sum().iloc[0])
print("Missing values after:", np.isnan(train_radius_imputed).sum())


### 5.2 KNN imputation example

`KNNImputer` estimates missing values using nearby samples.

This is more flexible than replacing every missing value with one global number, but it is also more computationally expensive.


In [ ]:
knn_features = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
]

knn_imputer = KNNImputer(n_neighbors=5)

X_knn = knn_imputer.fit_transform(X_train[knn_features])

print("Shape after KNN imputation:", X_knn.shape)
print("Remaining missing values:", np.isnan(X_knn).sum())


### 5.3 Other missing-value strategies
Compare row removal, column removal, and simple imputations without changing the main dataset. Deletion may lose many observations or introduce selection bias.

Forward/backward filling belongs to meaningfully ordered observations (for example, within a patient's time series), not an arbitrary list of unrelated patients. Backward filling can use future information and is unsuitable for real-time prediction.

KNN uses distances: different feature scales influence the neighbours. Consider scaling before KNN imputation, fitted only on training data.


In [ ]:
print("Training shape:", X_train.shape)
print("Complete rows:", X_train.dropna().shape)
print("Columns without missing values:", X_train.dropna(axis=1).shape)
for strategy in ["mean", "median", "most_frequent", "constant"]:
    imp = SimpleImputer(strategy=strategy, fill_value=0)
    values_demo = imp.fit_transform(X_train[["mean radius"]])
    print(strategy, "replacement:", imp.statistics_[0])
ordered_example = pd.Series([1.0, np.nan, 3.0, np.nan])
display(pd.DataFrame({
    "Original": ordered_example,
    "Forward fill": ordered_example.ffill(),
    "Backward fill": ordered_example.bfill(),
}))

# Scaling precedes KNN here; StandardScaler preserves NaNs until imputation.
scaled_knn = Pipeline([
    ("scale", StandardScaler()),
    ("impute", KNNImputer(n_neighbors=5)),
])
knn_train_scaled = scaled_knn.fit_transform(X_train[knn_features])
knn_test_scaled = scaled_knn.transform(X_test[knn_features])
print("Scaled KNN output:", knn_train_scaled.shape, knn_test_scaled.shape)


## 6. Outliers

An outlier may represent:

- a measurement or input error;
- corrupted data;
- a valid but unusual observation.

We should **identify outliers first**, not automatically delete every unusual value.

Two common methods are:

- standard deviation;
- interquartile range (IQR).


### 6.1 IQR method

A common rule marks values outside:

- `Q1 - 1.5 × IQR`
- `Q3 + 1.5 × IQR`


In [ ]:
feature = "mean radius"

values = X_train[feature].dropna()

q1 = values.quantile(0.25)
q3 = values.quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outlier_mask = (X_train[feature] < lower) | (X_train[feature] > upper)

print("Q1:", round(q1, 3))
print("Q3:", round(q3, 3))
print("IQR:", round(iqr, 3))
print("Lower limit:", round(lower, 3))
print("Upper limit:", round(upper, 3))
print("Identified outliers:", int(outlier_mask.sum()))

X_train.loc[outlier_mask, [feature]]


### 6.2 Standard-deviation method

Here we identify values farther than three standard deviations from the mean.


In [ ]:
feature = "mean texture"

values = X_train[feature].dropna()

mean_value = values.mean()
std_value = values.std()

lower = mean_value - 3 * std_value
upper = mean_value + 3 * std_value

outliers_std = values[(values < lower) | (values > upper)]

print("Mean:", round(mean_value, 3))
print("Standard deviation:", round(std_value, 3))
print("Lower limit:", round(lower, 3))
print("Upper limit:", round(upper, 3))
print("Identified outliers:", len(outliers_std))

outliers_std


### 6.3 Treating outliers: removal versus clipping
These are demonstrations on copies. The main model still uses the original training rows.
Estimate bounds from training data only. If removing training rows, apply the same mask to the target. Do not discard difficult test observations to improve a score.

Clipping replaces values beyond the bounds with the bounds; it does not remove rows. Neither clipping nor removal is automatically appropriate for valid extreme observations.


In [ ]:
outlier_features = ["mean radius", "mean texture", "mean perimeter", "mean area"]
q1 = X_train[outlier_features].quantile(0.25)
q3 = X_train[outlier_features].quantile(0.75)
iqr = q3 - q1
lower_bounds = q1 - 1.5 * iqr
upper_bounds = q3 + 1.5 * iqr
flagged = ((X_train[outlier_features] < lower_bounds) |
           (X_train[outlier_features] > upper_bounds)).any(axis=1)
X_train_filtered = X_train.loc[~flagged].copy()
y_train_filtered = y_train.loc[X_train_filtered.index]
assert X_train_filtered.index.equals(y_train_filtered.index)
print("IQR removal:", len(X_train), "->", len(X_train_filtered))

X_train_clipped = X_train.copy()
X_test_clipped = X_test.copy()
for col in outlier_features:
    X_train_clipped[col] = X_train[col].clip(lower_bounds[col], upper_bounds[col])
    X_test_clipped[col] = X_test[col].clip(lower_bounds[col], upper_bounds[col])
display(pd.DataFrame({
    "Before": X_train["mean radius"],
    "Clipped": X_train_clipped["mean radius"],
}).describe())

means = X_train[outlier_features].mean()
stds = X_train[outlier_features].std()
std_flags = ((X_train[outlier_features] - means).abs() > 3 * stds).any(axis=1)
print("Rows flagged by 3-SD rule:", int(std_flags.sum()))


### 6.4 Optional: Local Outlier Factor (LOF)
LOF compares local densities in a multivariate feature space. Impute and scale first; do not include the target or identifiers.
Here `contamination=0.05` sets an illustrative expected outlier fraction, not a discovered error rate. LOF labels are not proof that observations are incorrect.


In [ ]:
from sklearn.neighbors import LocalOutlierFactor

lof_preprocessor = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", RobustScaler()),
])
lof_input = lof_preprocessor.fit_transform(X_train[outlier_features])
lof_labels = LocalOutlierFactor(n_neighbors=20, contamination=0.05).fit_predict(lof_input)
lof_keep = pd.Series(lof_labels == 1, index=X_train.index)
X_train_lof = X_train.loc[lof_keep].copy()
y_train_lof = y_train.loc[lof_keep].copy()
print("LOF flagged rows:", int((~lof_keep).sum()))
display(y_train.value_counts().rename("Before").to_frame().join(
    y_train_lof.value_counts().rename("After LOF")))


## 7. Transforming numerical variables

The lecture introduces several common transformations:

- **Min-Max normalization** – usually maps values to `[0, 1]`;
- **standardization** – centers values around 0 and scales by standard deviation;
- **robust scaling** – uses the median and IQR and is less sensitive to outliers;
- **power transformation** – changes the distribution to be more Gaussian-like.

We fit every transformation on the **training data only**.


In [ ]:
feature = "mean radius"

# First remove missing values only for this small demonstration.
demo_imputer = SimpleImputer(strategy="median")
x = demo_imputer.fit_transform(X_train[[feature]])

scalers = {
    "MinMax": MinMaxScaler(),
    "Standard": StandardScaler(),
    "Robust": RobustScaler(),
}

scaled_versions = {}

for name, scaler in scalers.items():
    scaled_versions[name] = scaler.fit_transform(x).ravel()

comparison = pd.DataFrame({
    "Original": x.ravel(),
    "MinMax": scaled_versions["MinMax"],
    "Standard": scaled_versions["Standard"],
    "Robust": scaled_versions["Robust"],
})

comparison.describe().T


### Scaling formulas
Standardization: `z = (x - training_mean) / training_std`.
Min-Max: `z = (x - training_min) / (training_max - training_min)`.
Robust scaling: `z = (x - training_median) / training_IQR`.

Standardization does not require normally distributed inputs and does not make a skewed distribution normal. StandardScaler is sensitive to outliers. Min-Max scaling is also sensitive to extreme values; unseen test values can fall outside [0, 1]. RobustScaler reduces the influence of extremes on scaling parameters but does not remove outliers.


In [ ]:
# Match StandardScaler's population standard deviation (ddof=0).
manual_standard = (x - x.mean(axis=0)) / x.std(axis=0, ddof=0)
manual_minmax = (x - x.min(axis=0)) / (x.max(axis=0) - x.min(axis=0))
assert np.allclose(manual_standard.ravel(), scaled_versions["Standard"])
assert np.allclose(manual_minmax.ravel(), scaled_versions["MinMax"])
print("Manual formulas match scikit-learn.")


### 7.1 Power transformation example


In [ ]:
feature = "mean area"

area_imputer = SimpleImputer(strategy="median")
area = area_imputer.fit_transform(X_train[[feature]])

power = PowerTransformer()
area_power = power.fit_transform(area)

plt.figure(figsize=(7, 4))
plt.hist(area.ravel(), bins=30)
plt.title("Before PowerTransformer")
plt.xlabel(feature)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(area_power.ravel(), bins=30)
plt.title("After PowerTransformer")
plt.xlabel("Transformed value")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## 8. Categorical variables

This dataset contains two categorical variables:

- `device` – **nominal** categories with no natural order;
- `size_group` – **ordinal** categories: `small < medium < large`.

We therefore use:

- `OneHotEncoder` for `device`;
- `OrdinalEncoder` for `size_group`.


In [ ]:
# Nominal variable
nominal_imputer = SimpleImputer(strategy="most_frequent")
device_train = nominal_imputer.fit_transform(X_train[["device"]])

one_hot = OneHotEncoder(handle_unknown="ignore")
device_encoded = one_hot.fit_transform(device_train)

print("One-hot categories:")
print(one_hot.categories_)
print("Encoded shape:", device_encoded.shape)

# Ordinal variable
ordinal_imputer = SimpleImputer(strategy="most_frequent")
size_train = ordinal_imputer.fit_transform(X_train[["size_group"]])

ordinal_encoder = OrdinalEncoder(
    categories=[["small", "medium", "large"]],
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

size_encoded = ordinal_encoder.fit_transform(size_train)

print("\nFirst 10 ordinal values:")
print(size_encoded[:10].ravel())


## 9. Build one complete preprocessing pipeline

Instead of preprocessing each feature manually, we can combine the steps.

For this dataset:

- numerical variables → median imputation + `RobustScaler`;
- nominal categorical variable → most-frequent imputation + one-hot encoding;
- ordinal categorical variable → most-frequent imputation + ordinal encoding.

Then we train a Logistic Regression classifier.

Using a `Pipeline` is useful because the same preprocessing is applied consistently during training and testing.


In [ ]:
numeric_features = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
    "mean smoothness",
    "mean compactness",
    "mean concavity",
    "mean symmetry",
]

nominal_features = ["device"]
ordinal_features = ["size_group"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
])

nominal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=[["small", "medium", "large"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("nominal", nominal_pipeline, nominal_features),
    ("ordinal", ordinal_pipeline, ordinal_features),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Balanced accuracy:", round(balanced_accuracy_score(y_test, y_pred), 3))


In [ ]:
ConfusionMatrixDisplay.from_estimator(
    model,
    X_test,
    y_test,
    display_labels=["malignant", "benign"]
)
plt.title("Logistic Regression after data preparation")
plt.tight_layout()
plt.show()


## 10. Cross-validation and comparison of scalers

K-fold CV uses each fold once for validation and the remaining folds for training. StratifiedKFold approximately preserves class proportions.

Compare scalers with five-fold CV **within X_train only**, fitting the whole Pipeline in each fold. Report mean scores and fold-to-fold standard deviation (not a confidence interval). Select using CV, then evaluate on the held-out test subset. The introductory test result from Section 9 must not guide selection. Repeated choices based on test scores make the test set a validation set.

This example does not use the whole-dataset exploratory mRMR ranking.


In [ ]:
def build_model_with_scaler(scaler):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", scaler),
    ])

    nominal_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])

    ordinal_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(
            categories=[["small", "medium", "large"]],
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )),
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("nominal", nominal_pipeline, nominal_features),
        ("ordinal", ordinal_pipeline, ordinal_features),
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ])


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):
    print(f"Fold {fold}: train={len(train_idx)}, validation={len(val_idx)}")
scaler_options = {"MinMaxScaler": MinMaxScaler(),
                  "StandardScaler": StandardScaler(),
                  "RobustScaler": RobustScaler()}
results = []
for name, scaler in scaler_options.items():
    scores = cross_validate(build_model_with_scaler(clone(scaler)),
        X_train, y_train, cv=cv,
        scoring={"accuracy": "accuracy", "balanced_accuracy": "balanced_accuracy"},
        n_jobs=1, error_score="raise")
    results.append({"Scaler": name,
        "CV accuracy mean": scores["test_accuracy"].mean(),
        "CV balanced accuracy mean": scores["test_balanced_accuracy"].mean(),
        "CV balanced accuracy SD": scores["test_balanced_accuracy"].std(ddof=1)})
results_df = pd.DataFrame(results)
display(results_df.round(3))
best_name = results_df.loc[results_df["CV balanced accuracy mean"].idxmax(), "Scaler"]
final_model = build_model_with_scaler(clone(scaler_options[best_name]))
final_model.fit(X_train, y_train)
final_pred = final_model.predict(X_test)
print("Selected by training CV:", best_name)
print("Held-out accuracy:", round(accuracy_score(y_test, final_pred), 3))
print("Held-out balanced accuracy:", round(balanced_accuracy_score(y_test, final_pred), 3))


## 11. Feature selection with mRMR — whole-dataset demonstration

**mRMR (minimum Redundancy–Maximum Relevance)** selects features that are related to the target while limiting redundancy with already selected features.

Here we use **all samples in the cleaned dataset**, not only the training subset. This is an exploratory ranking demonstration. We do not train or report test accuracy for a model selected using this full-data ranking.

The train/test models in Sections 9–10 remain separate. For independent predictive evaluation, preprocessing and mRMR must instead be fitted within the training subset (or each cross-validation training fold).

We clone the preprocessor so that this demonstration does not change the fitted preprocessing of the earlier model. The candidate features are the numerical and categorical columns defined in Section 9; categorical columns are ranked after encoding.

This implementation uses the default F-statistic relevance and correlation-based redundancy. The displayed rank is the sequential selection order, not a p-value or a universal feature-importance score.


In [ ]:
# Full-dataset exploratory ranking: use all cleaned samples.
exploration_preprocessor = clone(preprocessor)
X_all_prepared = exploration_preprocessor.fit_transform(X)
if hasattr(X_all_prepared, "toarray"):
    X_all_prepared = X_all_prepared.toarray()

X_mrmr = pd.DataFrame(
    X_all_prepared,
    columns=exploration_preprocessor.get_feature_names_out(),
    index=X.index,
)
# Remove constant encoded columns, if any.
X_mrmr = X_mrmr.loc[:, X_mrmr.nunique() > 1]
y_mrmr = y.loc[X_mrmr.index]

# Request a ranking of all candidate columns.
ranked_features = mrmr_classif(
    X=X_mrmr,
    y=y_mrmr,
    K=X_mrmr.shape[1],
    n_jobs=1,
    show_progress=False,
)

ranking_df = pd.DataFrame({
    "Rank": np.arange(1, len(ranked_features) + 1),
    "Feature": ranked_features,
})
print("Samples used:", len(X_mrmr))
print("Candidate columns:", X_mrmr.shape[1])
display(ranking_df)

# Choose the first K features from the ranking.
K = 8
selected_features = ranked_features[:K]
X_selected = X_mrmr[selected_features].copy()
print(f"Selected {len(selected_features)} features (requested K={K}):")
print(selected_features)
print("Selected dataset shape:", X_selected.shape)


# Discussion questions for students

1. Why should duplicate samples be removed before splitting the data into training and test sets?
2. Why is a feature with only one unique value not useful for classification?
3. When can the median be a better imputation value than the mean?
4. What is the difference between a nominal and an ordinal categorical variable?
5. Why do we use `OneHotEncoder` for `device` but `OrdinalEncoder` for `size_group`?
6. Why should the scaler and imputer be fitted only on the training dataset?
7. Why can `RobustScaler` be useful when outliers are present?
8. Does an outlier always mean that the sample is wrong?
9. Why can different preprocessing methods lead to different model results?

10. How does mRMR differ from selecting features based only on their individual relationship with the target?
11. Why is the whole-dataset mRMR ranking in Section 11 an exploratory result rather than an independent predictive evaluation?


## 12. Student experiment

Change **one thing at a time**, rerun the model, and record what happens.

### Experiment A – imputation

Replace:

```python
SimpleImputer(strategy="median")
```

with:

```python
SimpleImputer(strategy="mean")
```

Does the result change?

### Experiment B – scaling

Compare:

```python
MinMaxScaler()
StandardScaler()
RobustScaler()
```

Which has the highest mean CV balanced accuracy? Compare fold-to-fold variability too.

### Experiment C – mRMR feature selection

Use the whole-dataset ranking from Section 11 and compare subsets:

```python
for k in [5, 8, 10]:
    selected = ranked_features[:k]
    print(f"K={k}: {selected}")
```

Which features are selected first? Do the selected features contain similar information?
Inspect their pairwise correlations using:

```python
X_mrmr[ranked_features[:8]].corr()
```

This experiment compares exploratory feature subsets, not independent test accuracy.

### Experiment D – train/test split

Change:

```python
test_size=0.30
```

to `0.20` or `0.40`.

What changes and why?

### Main conclusion

> Data preparation is not one fixed recipe. The appropriate choices depend on the data, missing values, outliers, feature types, and the machine-learning method.


## 13. Export prepared datasets and results

Use the final model's training-fitted preprocessor. Export training and test data separately, preserving row identifiers and target alignment. Encoded/scaled values are not original measurement units.

The whole-dataset mRMR exports are labelled exploratory and are not used in the independent test evaluation.


In [ ]:
export_preprocessor = final_model.named_steps["preprocessor"]
export_names = export_preprocessor.get_feature_names_out()
def prepared_frame(frame):
    matrix = export_preprocessor.transform(frame)
    if hasattr(matrix, "toarray"):
        matrix = matrix.toarray()
    return pd.DataFrame(matrix, columns=export_names, index=frame.index)
prepared_train = prepared_frame(X_train)
prepared_test = prepared_frame(X_test)
prepared_train["target"] = y_train
prepared_test["target"] = y_test
assert not prepared_train.isna().any().any()
assert not prepared_test.isna().any().any()
assert prepared_train.index.intersection(prepared_test.index).empty
prepared_train.to_csv(OUTPUT_DIR / "prepared_train.csv", index_label="row_id")
prepared_test.to_csv(OUTPUT_DIR / "prepared_test.csv", index_label="row_id")
results_df.to_csv(OUTPUT_DIR / "scaler_cross_validation.csv", index=False)
ranking_df.to_csv(OUTPUT_DIR / "mrmr_exploratory_ranking.csv", index=False)
exploratory_export = X_selected.copy()
exploratory_export["target"] = y_mrmr
exploratory_export.to_csv(OUTPUT_DIR / "mrmr_exploratory_selected.csv", index_label="row_id")
for name, matrix in correlation_matrices.items():
    matrix.to_csv(OUTPUT_DIR / f"correlation_{name}.csv")
print("Saved outputs:", sorted(p.name for p in OUTPUT_DIR.iterdir()))


## 14. Independent assignment and assessment

Prepare the course dataset independently, or use Titanic / another suitable labelled tabular dataset. Adapt feature names, target and encoders to your dataset. Do not use identifiers as predictors.

Submit a runnable notebook explaining your decisions, prepared train/test CSVs, the exploratory mRMR ranking, a CV results table, and a short conclusion. Include a histogram, scatter plot, group boxplot and correlation heatmap.

| Criterion | Points |
|---|---:|
| Inspect types, missingness, duplicates and distributions; demonstrate pandas selection/editing | 2 |
| Compare missing-value strategies and justify your choice | 1 |
| Calculate Pearson, Spearman and Kendall correlations; interpret association and redundancy | 2 |
| Explain and apply scaling and categorical encoding | 1 |
| Detect outliers with IQR and SD; demonstrate treatment and justify whether to retain it | 2 |
| Use stratified splitting and pipeline CV; distinguish exploratory mRMR from independent evaluation | 1 |
| Export aligned data/results and provide a concise conclusion | 1 |
| **Total** | **10** |

Optional: compare LOF with IQR and SD, or repeat on a second dataset. More removed rows does not mean a better method.

### Final questions
1. How much data does complete-case deletion discard?
2. Why might Pearson and Spearman differ for the same pair?
3. Why must X and y use the same mask after row removal?
4. Does clipping change the number of samples?
5. How does mRMR account for redundancy?
6. How does a CV validation fold differ from the final test set?
7. Why should the full-data exploratory ranking not guide independent test evaluation?

### Documentation
- [pandas indexing](https://pandas.pydata.org/docs/user_guide/indexing.html)
- [Correlation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html)
- [Preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html)
- [Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html)
- [LOF](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.LocalOutlierFactor.html)
- [mrmr-selection](https://github.com/smazzanti/mrmr)
